<a href="https://colab.research.google.com/github/gseetharami352005/CSA6102/blob/main/exp10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================
# Email Header Analyzer for Spoofed Sender Detection
# Google Colab Ready
# ============================================

import re
from email import policy
from email.parser import BytesParser
from google.colab import files

# Step 1: Upload raw email header file
print("Upload a raw email header file (.eml or .txt):")
uploaded = files.upload()

email_file = list(uploaded.keys())[0]

# Step 2: Read the raw email
with open(email_file, "rb") as f:
    raw_email = f.read()

# Step 3: Parse the email
msg = BytesParser(policy=policy.default).parsebytes(raw_email)

# Step 4: Extract important headers
from_header = msg.get("From", "Not Found")
reply_to = msg.get("Reply-To", "Not Found")
return_path = msg.get("Return-Path", "Not Found")
message_id = msg.get("Message-ID", "Not Found")
received_headers = msg.get_all("Received", [])

print("\n====================================")
print("EMAIL HEADER ANALYSIS")
print("====================================")

print("\nFrom:", from_header)
print("Reply-To:", reply_to)
print("Return-Path:", return_path)
print("Message-ID:", message_id)

# Step 5: Display Received headers
print("\nReceived Headers:")
for i, received in enumerate(received_headers, 1):
    print(f"{i}. {received}")

# Step 6: Extract email domains
def extract_domain(email_address):
    match = re.search(r'@([A-Za-z0-9.-]+)', email_address)
    if match:
        return match.group(1).lower()
    return None

from_domain = extract_domain(from_header)
reply_domain = extract_domain(reply_to)
return_domain = extract_domain(return_path)

print("\n====================================")
print("SPOOFING INDICATOR ANALYSIS")
print("====================================")

spoofing_detected = False

# Check 1: From vs Reply-To domain
if from_domain and reply_domain:
    if from_domain != reply_domain:
        print("[WARNING] From and Reply-To domains do not match.")
        spoofing_detected = True
    else:
        print("[OK] From and Reply-To domains match.")

# Check 2: From vs Return-Path domain
if from_domain and return_domain:
    if from_domain != return_domain:
        print("[WARNING] From and Return-Path domains do not match.")
        spoofing_detected = True
    else:
        print("[OK] From and Return-Path domains match.")

# Check 3: Authentication results
authentication = msg.get("Authentication-Results", "")

print("\nAuthentication Results:")
if authentication:
    print(authentication)

    if "spf=fail" in authentication.lower():
        print("[WARNING] SPF authentication failed.")
        spoofing_detected = True

    if "dkim=fail" in authentication.lower():
        print("[WARNING] DKIM authentication failed.")
        spoofing_detected = True

    if "dmarc=fail" in authentication.lower():
        print("[WARNING] DMARC authentication failed.")
        spoofing_detected = True

else:
    print("[INFO] No Authentication-Results header found.")

# Final result
print("\n====================================")
print("FINAL RESULT")
print("====================================")

if spoofing_detected:
    print("[ALERT] Possible spoofed sender detected.")
else:
    print("[INFO] No obvious spoofing indicators detected.")

Upload a raw email header file (.eml or .txt):


Saving archive.zip to archive.zip

EMAIL HEADER ANALYSIS

From: Not Found
Reply-To: Not Found
Return-Path: Not Found
Message-ID: Not Found

Received Headers:

SPOOFING INDICATOR ANALYSIS

Authentication Results:
[INFO] No Authentication-Results header found.

FINAL RESULT
[INFO] No obvious spoofing indicators detected.
